# Symbolic student — expert-count ablation (pure regression, self-signed, calibrated)

Trains a symbolic physics student with `N_EXPERTS ∈ {1, 2, 4}` and dumps a calibrated parquet
that the comparison notebook reads directly. **No teacher, no KD** — pure regression on the
packed-14 output.

**To run an ablation point:** set `N_EXPERTS` in the config cell, then *Restart & Run All*.
Each value writes to its own `symbolic_ablation_nexp{N}/` directory.

### The three fixes baked in (learned the hard way; do not remove)
1. **Axis swap** — the ansatz builds its x-mean from rows (which track true *y*) and vice-versa.
   `AxisFixExpert` transposes the two position slots back. Baked into `make_expert`, so
   *Run-All can never strip it.*
2. **Two-phase means training** — plain NLL collapses by inflating σ (kills the gradient on the
   means). Phase 1 fits means under MSE; phase 2 refines under NLL.
3. **Corrected error-head loss + matched decode** — `loss.custom_loss` uses `sum` + `relu`
   diagonal + a `clip` floor that zeros gradients on floored events, leaving σ untrained
   (esp. σ_x, which floors to ~0). Phase 3 retrains **only** the error head under
   `mean(-log_prob)` with a **softplus** diagonal; the parquet dump decodes σ with the *same*
   softplus so pulls land at ~1.

Prereq: the TimeGrad `ansatz.py` must be on disk (cell 1 of `distill_moe_timegrad_sign.ipynb`,
then restart kernel). Cell 1 below asserts this.

In [1]:
# ---- 1. setup ----
import os
WORKDIR = "/depot/cms/private/users/kuang14/Smart_Pixel/smart-pixels-ml_symbolic"
HELPERS = os.path.join(WORKDIR, "two_bit_optimization_helpers")
os.chdir(WORKDIR)

import sys, json, glob, time
sys.path.insert(0, HELPERS)

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp

for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass

from prepare_tfrecords import generate_tfrecords, load_tfrecords
from loss import custom_loss                       # original loss: used only for the init smoke-test
from models.student_max import build_student_max, pack_14

# the TimeGrad ansatz (self-signed) must be the one on disk
assert "sign_k_alpha" in open(os.path.join(HELPERS, "symbolic", "ansatz.py")).read(), \
       "TimeGrad ansatz NOT on disk -- run cell 1 of distill_moe_timegrad_sign.ipynb, then restart kernel"
print("TF", tf.__version__, "| TimeGrad ansatz on disk: OK | GPUs:", tf.config.list_physical_devices("GPU"))
pi = np.pi

2026-07-13 00:10:21.496236: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-13 00:10:21.496314: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-13 00:10:21.497501: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-13 00:10:21.505992: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-13 00:10:26.604974: W tensorflow/compiler/tf2

TF 2.15.1 | TimeGrad ansatz on disk: OK | GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ---- 2. config ----   *** set N_EXPERTS here, then Restart & Run All ***
SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED)

DATASET_DIR      = "/depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets"
SELECT_CONTAINED = True
TIMESLICES       = 2
BATCH            = 5000
TFRECORDS_EXIST  = True

VARIANT   = "barycenter"

# ======== THE ABLATION KNOB ========
N_EXPERTS = 4                     # 1 (single formula), 2, or 4
# router exists only for N_EXPERTS >= 2; kept small so it doesn't dominate the param budget
ROUTER_HIDDEN = {4: (32, 16), 2: (16,)}.get(N_EXPERTS, ())
ROUTER_TEMP   = 1.0               # raise to 2.0 if usage collapses onto one expert
EXPERT_KWARGS = {}

# inputs: clean analog (NO noise, NO digitize) -- locked
NOISE    = -1
DIGITIZE = False

# training schedule
LR        = 1e-3
EPOCHS_P1 = 150       # phase 1: means (MSE)
EPOCHS_P2 = 200       # phase 2: means+cov refine (original NLL)
EPOCHS_P3 = 120       # phase 3: error head only (corrected NLL)

OUT_DIR = f"/depot/cms/private/users/kuang14/Smart_Pixel/symbolic_ablation_nexp{N_EXPERTS}"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"N_EXPERTS={N_EXPERTS} | ROUTER_HIDDEN={ROUTER_HIDDEN} | OUT_DIR: {OUT_DIR}")

N_EXPERTS=4 | ROUTER_HIDDEN=(32, 16) | OUT_DIR: /depot/cms/private/users/kuang14/Smart_Pixel/symbolic_ablation_nexp4


In [3]:
# ---- 3. data (analog, clean) ----
_, _, tfr_tr, tfr_val = generate_tfrecords(
    dataset_dir=DATASET_DIR, model_type="ViT_Max",     # tfrecord format only; NO teacher is loaded
    train_batch_size=BATCH, val_batch_size=BATCH,
    select_contained=SELECT_CONTAINED, timeslices=TIMESLICES,
    tfrecords_exist=TFRECORDS_EXIST, seed=SEED,
)
tg, vg = load_tfrecords(tfr_tr, tfr_val, noise=NOISE, digitize=DIGITIZE, seed=SEED)

labels_scale = json.load(open(os.path.join(tfr_tr, "metadata.json")))["labels_scale"]
print("labels_scale:", labels_scale, "| train batches:", len(tg), "| val batches:", len(vg))

xb, yb = tg[0]; xb = np.asarray(xb); yb = np.asarray(yb)
assert xb.shape[1:] == (16, 16, 2) and yb.shape[1] == 4
assert NOISE == -1 and DIGITIZE is False, "input config drifted -- locked to clean analog"
print("x:", xb.shape, "| y:", yb.shape)

Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_train_contained/metadata.json


Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_test_contained/metadata.json
Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_train_contained/metadata.json


Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_test_contained/metadata.json
labels_scale: [123.41016201181557, 30.929849811025303, 6.577498885094723, 1.9295648020239338] | train batches: 87 | val batches: 22


2026-07-13 00:10:38.438804: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1775 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB MIG 1g.5gb, pci bus id: 0000:81:00.0, compute capability: 8.0


x: (5000, 16, 16, 2) | y: (5000, 4)


In [4]:
# ---- 4. calibrate the TimeGrad sign gates from TRAIN data (exact k, a0 inits) ----
# 1-feature logistic  P(sign=+) = sigma(w*T + b)  maps to  tanh(k*(a0 - T)),  k = -w/2, a0 = -b/w.
# alpha sign <- Ty (between-slice y-centroid drift); beta sign <- Tx. Units: um. Fit TRAIN, verify VAL.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

def drifts_np(X):                               # X (N,16,16,2) -> Tx, Ty in um
    q = np.maximum(np.asarray(X, "float32"), 0.0)
    gx, gy = np.meshgrid(np.arange(16), np.arange(16), indexing="ij")
    xw = (gx * 50.0)[None]; yw = (gy * 12.5)[None]
    q0 = q[..., 0].sum((1, 2)) + 1e-6; q1 = q[..., 1].sum((1, 2)) + 1e-6
    Tx = (q[..., 1] * xw[0]).sum((1, 2)) / q1 - (q[..., 0] * xw[0]).sum((1, 2)) / q0
    Ty = (q[..., 1] * yw[0]).sum((1, 2)) / q1 - (q[..., 0] * yw[0]).sum((1, 2)) / q0
    return Tx, Ty

def collect(gen, n_batches=None):
    n = len(gen) if n_batches is None else min(n_batches, len(gen))
    Xs, Ys = zip(*[(np.asarray(gen[i][0]), np.asarray(gen[i][1])) for i in range(n)])
    return np.concatenate(Xs), np.concatenate(Ys)

Xtr_, Ytr_ = collect(tg, n_batches=8)
Xva_, Yva_ = collect(vg)
Tx_tr, Ty_tr = drifts_np(Xtr_); Tx_va, Ty_va = drifts_np(Xva_)

SIGN_INIT = {}
for nm, T_tr, T_va, i in [("alpha", Ty_tr, Ty_va, 2), ("beta", Tx_tr, Tx_va, 3)]:
    ys_tr = (Ytr_[:, i] > 0).astype(int)
    clf = LogisticRegression(max_iter=5000).fit(T_tr.reshape(-1, 1), ys_tr)
    w, b = float(clf.coef_[0, 0]), float(clf.intercept_[0])
    k, a0 = -w / 2.0, -b / w
    hard = np.where(a0 - T_va > 0, 1.0, -1.0)
    agr  = (hard == np.sign(Yva_[:, i])).mean()
    bal  = balanced_accuracy_score((Yva_[:, i] > 0).astype(int), (hard > 0).astype(int))
    SIGN_INIT[f"initial_sign_k_{nm}"]  = k
    SIGN_INIT[f"initial_sign_a0_{nm}"] = a0
    print(f"cot{nm}: k={k:+.4f}/um  a0={a0:+.3f} um  | VAL raw agree {agr:.4f}  bal-acc {bal:.4f}")
    assert bal > 0.90, f"cot{nm} calibration failed -- STOP, do not train"
print("\nSIGN_INIT:", {k: round(v, 4) for k, v in SIGN_INIT.items()})
json.dump(SIGN_INIT, open(os.path.join(OUT_DIR, "sign_init.json"), "w"), indent=1)
del Xtr_, Ytr_

cotalpha: k=+0.6289/um  a0=+0.038 um  | VAL raw agree 0.9672  bal-acc 0.9672
cotbeta: k=+0.0836/um  a0=-43.429 um  | VAL raw agree 0.9645  bal-acc 0.9631

SIGN_INIT: {'initial_sign_k_alpha': 0.6289, 'initial_sign_a0_alpha': 0.0376, 'initial_sign_k_beta': 0.0836, 'initial_sign_a0_beta': -43.4286}


In [5]:
# ---- 5. student model  (AXIS FIX BAKED IN -- Run-All safe) ----
class RouterFeatures(tf.keras.layers.Layer):
    # HARDENED: relu charge, std floored, finite-guard + clip.
    def call(self, x):
        xp  = tf.nn.relu(x)
        q   = tf.reduce_sum(xp, axis=-1)
        px  = tf.reduce_sum(q, axis=2)
        py  = tf.reduce_sum(q, axis=1)
        tot = tf.reduce_sum(py, axis=1, keepdims=True) + 1e-9
        pxn, pyn = px / tot, py / tot
        yc  = tf.range(16, dtype=tf.float32) - 7.5
        mu  = tf.reduce_sum(pyn * yc, axis=1, keepdims=True)
        d   = yc[None, :] - mu
        m2  = tf.reduce_sum(pyn * d**2, axis=1, keepdims=True)
        m3  = tf.reduce_sum(pyn * d**3, axis=1, keepdims=True)
        std = tf.sqrt(m2 + 1.0)
        skew = m3 / (std ** 3)
        py0 = tf.reduce_sum(xp[..., 0],  axis=1)
        py1 = tf.reduce_sum(xp[..., -1], axis=1)
        c0  = tf.reduce_sum(py0*yc, 1, keepdims=True) / (tf.reduce_sum(py0, 1, keepdims=True) + 1e-9)
        c1  = tf.reduce_sum(py1*yc, 1, keepdims=True) / (tf.reduce_sum(py1, 1, keepdims=True) + 1e-9)
        tasym = c1 - c0
        logq  = tf.math.log(tot + 1.0)
        f = tf.concat([pxn, pyn, std, skew, tasym, logq], axis=-1)
        f = tf.where(tf.math.is_finite(f), f, tf.zeros_like(f))
        return tf.clip_by_value(f, -8.0, 8.0)

class AxisFixExpert(tf.keras.Model):
    # ansatz emits position slot0 from ROWS (=true y) and slot1 from COLS (=true x): transposed.
    # Swap the two position means back. Angles (slots 2,3), chol, and all sign gates untouched.
    def __init__(self, inner, **kw):
        super().__init__(**kw)
        self.inner = inner
    def call(self, x, training=False):
        means, chol = self.inner(x, training=training)
        means = tf.concat([means[:, 1:2], means[:, 0:1], means[:, 2:]], axis=-1)  # x <-> y
        return means, chol
    def get_layer(self, *a, **k):        # expose inner physics_ansatz to later cells
        return self.inner.get_layer(*a, **k)

def make_expert():
    inner = build_student_max(VARIANT,
                              ansatz_kwargs={"labels_scale": labels_scale, **SIGN_INIT},
                              **EXPERT_KWARGS)
    return AxisFixExpert(inner)

class MoECore(tf.keras.Model):
    def __init__(self, n_experts, router_hidden, temp, **kw):
        super().__init__(**kw)
        self.n_experts, self.temp = n_experts, temp
        self.feats  = RouterFeatures(name="router_features")
        self.router = tf.keras.Sequential(
            [tf.keras.layers.Dense(h, activation="relu") for h in router_hidden]
            + [tf.keras.layers.Dense(n_experts)], name="router")
        self.experts = [make_expert() for _ in range(n_experts)]
    def route(self, x):
        return tf.nn.softmax(self.router(self.feats(x)) / self.temp, axis=-1)
    def call(self, x, training=False):
        w = self.route(x)
        outs  = [e(x, training=training) for e in self.experts]
        means = tf.add_n([w[:, k:k+1] * outs[k][0] for k in range(self.n_experts)])
        chol  = tf.add_n([w[:, k:k+1] * outs[k][1] for k in range(self.n_experts)])
        return means, chol

class SingleCore(tf.keras.Model):                 # N_EXPERTS == 1: no router at all
    def __init__(self, **kw):
        super().__init__(**kw)
        self.experts = [make_expert()]
    def call(self, x, training=False):
        return self.experts[0](x, training=training)

class PackedStudent(tf.keras.Model):              # (mu, chol) -> packed 14 for the loss
    def __init__(self, core, **kw):
        super().__init__(**kw)
        self.core = core
    def call(self, x, training=False):
        mu, chol = self.core(x, training=training)
        return pack_14(mu, chol)

core  = SingleCore(name="single_core") if N_EXPERTS == 1 else \
        MoECore(N_EXPERTS, ROUTER_HIDDEN, ROUTER_TEMP, name="moe_core")
model = PackedStudent(core, name=f"symbolic_n{N_EXPERTS}")
assert model(xb[:64]).shape[-1] == 14

n_router = core.router.count_params() if N_EXPERTS > 1 else 0
print(f"total params: {model.count_params()} | router: {n_router} "
      f"| per-expert: {core.experts[0].count_params()}")

# TimeGrad sign scalars must be live in EVERY expert
names = [v.name for v in model.trainable_variables]
for need in ("sign_k_alpha", "sign_a0_alpha", "sign_k_beta", "sign_a0_beta"):
    n_found = sum(need in n for n in names)
    assert n_found == N_EXPERTS, f"{need}: found {n_found}, expected {N_EXPERTS} -- stale ansatz.py / kernel not restarted"
print("TimeGrad sign scalars present in all", N_EXPERTS, "expert(s)")

2026-07-13 00:10:45.647947: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


total params: 3268 | router: 1780 | per-expert: 372
TimeGrad sign scalars present in all 4 expert(s)


2026-07-13 00:10:48.010567: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


In [6]:
# ---- 6. pre-train checks: axis wiring + init sign agreement + NaN scan ----
cb, lb = np.asarray(vg[0][0]), np.asarray(vg[0][1])
mu0 = np.asarray(core(cb, training=False)[0])

# (a) axis fix verified: x-slot tracks true_x, y-slot tracks true_y (both ~0.92)
for nm, i in [("x", 0), ("y", 1)]:
    cx = np.corrcoef(mu0[:, i], lb[:, 0])[0, 1]; cy = np.corrcoef(mu0[:, i], lb[:, 1])[0, 1]
    print(f"  {nm}-slot: corr(true_x)={cx:+.3f}  corr(true_y)={cy:+.3f}")
    assert abs([cx, cy][i]) > 0.8, f"{nm} not on its own axis -- axis fix broken, STOP"

# (b) init sign agreement (aff=(1,0) at init, so sign(cot slot) == ansatz TimeGrad sign)
for nm, i in [("cotA", 2), ("cotB", 3)]:
    m = mu0[:, i] != 0
    agr = (np.sign(mu0[m, i]) == np.sign(lb[m, i])).mean()
    print(f"  {nm}: init sign agreement {agr:.4f}  (on {m.mean():.1%} nonzero-|cot| events)")
    assert agr > 0.90, f"{nm} init sign broken -- do NOT train"

# (c) init loss finite + full-val NaN scan
l0 = float(tf.reduce_mean(custom_loss(tf.constant(yb, tf.float32), model(xb, training=False))))
print("  init custom_loss:", round(l0, 4)); assert np.isfinite(l0)
bad = sum(0 if np.isfinite(model(vg[i][0], training=False).numpy()).all() else 1 for i in range(len(vg)))
print("  NaN scan | bad val batches:", bad, "/", len(vg)); assert bad == 0
if N_EXPERTS > 1:
    print("  router usage (untrained, ~%.2f each):" % (1 / N_EXPERTS), core.route(xb).numpy().mean(0).round(3))
print("all pre-train checks passed")

  x-slot: corr(true_x)=+0.939  corr(true_y)=+0.006
  y-slot: corr(true_x)=-0.003  corr(true_y)=+0.922
  cotA: init sign agreement 0.9642  (on 100.0% nonzero-|cot| events)
  cotB: init sign agreement 0.9652  (on 100.0% nonzero-|cot| events)


2026-07-13 00:10:48.434089: I tensorflow/core/util/cuda_solvers.cc:179] Creating GpuSolver handles for stream 0x563fa1469810


  init custom_loss: 98788.3281
  NaN scan | bad val batches: 0 / 22
  router usage (untrained, ~0.25 each): [0.605 0.144 0.09  0.162]
all pre-train checks passed


In [7]:
# ---- 7. train phase 1 (means, MSE) + phase 2 (means+cov, original NLL) ----
class NaNStop(tf.keras.callbacks.Callback):
    def on_train_batch_end(self, b, logs=None):
        v = (logs or {}).get("loss")
        if v is not None and not np.isfinite(v):
            print(f"\nNaN loss at batch {b} -- stopping"); self.model.stop_training = True

def mse_means(y_true, y_pred14):
    mu = tf.gather(y_pred14, [0, 2, 4, 6], axis=-1)
    return tf.reduce_mean(tf.square(y_true - mu), axis=-1)

# phase 1: means only. Prevents the NLL sigma-inflation collapse (which kills the mean gradient).
model.compile(optimizer=tf.keras.optimizers.Adam(LR, clipnorm=1.0), loss=mse_means)
t0 = time.time()
h1 = model.fit(tg, validation_data=vg, epochs=EPOCHS_P1, shuffle=False, verbose=1,
               callbacks=[tf.keras.callbacks.CSVLogger(os.path.join(OUT_DIR, "history_p1.csv")),
                          NaNStop(),
                          tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=30,
                                                           restore_best_weights=True, verbose=1)])
print("phase 1 done in %.0fs" % (time.time() - t0))

# means must be alive before phase 2
mu_v = np.asarray(core(np.asarray(vg[0][0]), training=False)[0]); yv = np.asarray(vg[0][1])
for nm, i in [("x", 0), ("y", 1)]:
    r = np.corrcoef(mu_v[:, i], yv[:, i])[0, 1]
    print(f"phase-1 {nm}: corr(pred, true) = {r:.4f}")
    assert r > 0.5, f"{nm} means still dead after phase 1 -- STOP"

# phase 2: joint refine under the ORIGINAL loss at lower LR (means settle with cov present)
model.compile(optimizer=tf.keras.optimizers.Adam(LR / 3, clipnorm=1.0), loss=custom_loss)
t0 = time.time()
h2 = model.fit(tg, validation_data=vg, epochs=EPOCHS_P2, shuffle=False, verbose=1,
               callbacks=[tf.keras.callbacks.CSVLogger(os.path.join(OUT_DIR, "history_p2.csv")),
                          NaNStop(),
                          tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=50,
                                                           restore_best_weights=True, verbose=1)])
print("phase 2 done in %.0fs" % (time.time() - t0))

Epoch 1/150


2026-07-13 00:10:57.125673: I external/local_xla/xla/service/service.cc:168] XLA service 0x7ef4edc4ee20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-07-13 00:10:57.125733: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-PCIE-40GB MIG 1g.5gb, Compute Capability 8.0
2026-07-13 00:10:57.132142: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1783894257.227824   53937 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


87/87 [==============================] - 17s 91ms/step - loss: 0.1847 - val_loss: 0.1294
Epoch 2/150
87/87 [==============================] - 7s 86ms/step - loss: 0.1073 - val_loss: 0.0912
Epoch 3/150
87/87 [==============================] - 7s 86ms/step - loss: 0.0798 - val_loss: 0.0707
Epoch 4/150
87/87 [==============================] - 7s 85ms/step - loss: 0.0624 - val_loss: 0.0552
Epoch 5/150
87/87 [==============================] - 7s 86ms/step - loss: 0.0506 - val_loss: 0.0474
Epoch 6/150
87/87 [==============================] - 8s 86ms/step - loss: 0.0446 - val_loss: 0.0428
Epoch 7/150
87/87 [==============================] - 7s 84ms/step - loss: 0.0409 - val_loss: 0.0398
Epoch 8/150
87/87 [==============================] - 7s 84ms/step - loss: 0.0385 - val_loss: 0.0378
Epoch 9/150
87/87 [==============================] - 7s 85ms/step - loss: 0.0367 - val_loss: 0.0363
Epoch 10/150
87/87 [==============================] - 7s 84ms/step - loss: 0.0353 - val_loss: 0.0350
Epoch 11/1

In [8]:
# ---- 8. train phase 3: error head ONLY, corrected & stable NLL ----
# WHY: loss.custom_loss uses K.sum (batch-size-dependent LR) + a relu diagonal + a clip floor
# that zeros gradients on floored events -> sigma never trains (sigma_x floors to ~0). Here we
# freeze the means (physics ansatz) and train only the error MLP under mean(-log_prob) with a
# SOFTPLUS diagonal (smooth, never hard-zeros the gradient). The dump in cell 10 uses the SAME
# softplus decode, so pulls land at ~1.
def custom_loss_fixed(y, p):
    mu   = p[:, 0:8:2]
    Mdia = 1e-9 + tf.nn.softplus(p[:, 1:8:2])
    Mcov = p[:, 8:]
    z = tf.zeros_like(Mdia[:, 0])
    L = tf.transpose(tf.stack([
        tf.stack([Mdia[:, 0], z,          z,          z         ]),
        tf.stack([Mcov[:, 0], Mdia[:, 1], z,          z         ]),
        tf.stack([Mcov[:, 1], Mcov[:, 2], Mdia[:, 2], z         ]),
        tf.stack([Mcov[:, 3], Mcov[:, 4], Mcov[:, 5], Mdia[:, 3]]),
    ]), perm=[2, 0, 1])
    dist = tfp.distributions.MultivariateNormalTriL(loc=mu, scale_tril=L)
    return tf.reduce_mean(-dist.log_prob(y))

# freeze the physics ansatz (means); only the error MLP stays trainable
for e in core.experts:
    e.get_layer("physics_ansatz").trainable = False
trainable = [v.name for v in model.trainable_variables]
print("trainable now (expect only error_mlp / chol_entries / hidden):")
for n in trainable: print("  ", n)

model.compile(optimizer=tf.keras.optimizers.Adam(3e-4, clipnorm=1.0), loss=custom_loss_fixed)
t0 = time.time()
h3 = model.fit(tg, validation_data=vg, epochs=EPOCHS_P3, shuffle=False, verbose=1,
               callbacks=[tf.keras.callbacks.CSVLogger(os.path.join(OUT_DIR, "history_p3.csv")),
                          NaNStop(),
                          tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=25,
                                                           restore_best_weights=True, verbose=1)])
print("phase 3 done in %.0fs (val_loss should be NEGATIVE)" % (time.time() - t0))
model.save_weights(os.path.join(OUT_DIR, f"symbolic_n{N_EXPERTS}.weights.h5"))
print("saved weights ->", os.path.join(OUT_DIR, f"symbolic_n{N_EXPERTS}.weights.h5"))

trainable now (expect only error_mlp / chol_entries / hidden):
   dense/kernel:0
   dense/bias:0
   dense_1/kernel:0
   dense_1/bias:0
   dense_2/kernel:0
   dense_2/bias:0
   hidden/kernel:0
   hidden/bias:0
   chol_entries/kernel:0
   chol_entries/bias:0
   hidden/kernel:0
   hidden/bias:0
   chol_entries/kernel:0
   chol_entries/bias:0
   hidden/kernel:0
   hidden/bias:0
   chol_entries/kernel:0
   chol_entries/bias:0
   hidden/kernel:0
   hidden/bias:0
   chol_entries/kernel:0
   chol_entries/bias:0
Epoch 1/120
87/87 [==============================] - 13s 93ms/step - loss: 2.0020 - val_loss: 1.6542
Epoch 2/120
87/87 [==============================] - 7s 84ms/step - loss: 1.3131 - val_loss: 0.9672
Epoch 3/120
87/87 [==============================] - 7s 84ms/step - loss: 0.6259 - val_loss: 0.2761
Epoch 4/120
87/87 [==============================] - 7s 84ms/step - loss: -0.0947 - val_loss: -0.4713
Epoch 5/120
87/87 [==============================] - 7s 83ms/step - loss: -0.8515 - val_

saved weights -> /depot/cms/private/users/kuang14/Smart_Pixel/symbolic_ablation_nexp4/symbolic_n4.weights.h5


In [9]:
# ---- 9. post-train: sign agreement + residual summary (physical units) ----
ls = np.asarray(labels_scale, dtype=float)
P, Yt = [], []
for i in range(len(vg)):
    cb, lb = vg[i]
    s14 = model(cb, training=False).numpy()
    P.append(s14[:, [0, 2, 4, 6]]); Yt.append(np.asarray(lb))
P = np.concatenate(P); Yt = np.concatenate(Yt)

for nm, i in [("cotA", 2), ("cotB", 3)]:
    ti = np.sign(Yt[:, i]); pr = np.sign(P[:, i]); m = P[:, i] != 0
    conf = np.abs(Yt[:, i]) > np.quantile(np.abs(Yt[:, i]), 0.5)
    print(f"{nm} sign agreement: {(pr[m]==ti[m]).mean():.4f}  (confident half: {(pr[m&conf]==ti[m&conf]).mean():.4f})")

for nm, i in [("x", 0), ("y", 1)]:
    r = (Yt[:, i] - P[:, i]) * ls[i]
    print(f"{nm:<6} mean {r.mean():8.4f}  std {r.std():8.4f} um")

def cot_to_deg(c): return np.degrees(np.arctan2(1.0, c))
for nm, i in [("cotA", 2), ("cotB", 3)]:
    r = cot_to_deg(Yt[:, i] * ls[i]) - cot_to_deg(P[:, i] * ls[i])
    print(f"{nm:<6} mean {np.mean(r):8.4f}  std {np.std(r):8.4f} deg")

cotA sign agreement: 0.9727  (confident half: 0.9996)
cotB sign agreement: 0.9672  (confident half: 0.9991)
x      mean   0.1749  std   8.2681 um
y      mean   0.0530  std   4.7153 um
cotA   mean   0.3821  std   5.8611 deg
cotB   mean  -0.2028  std   5.4835 deg


In [10]:
# ---- 10. calibrated parquet dump  (softplus decode -- MUST match custom_loss_fixed) ----
softplus = lambda z: np.log1p(np.exp(-np.abs(z))) + np.maximum(z, 0.0)   # stable, == tf.nn.softplus

P, Yt, S = [], [], []
for i in range(len(vg)):
    cb, lb = vg[i]
    s14 = model(cb, training=False).numpy()
    P.append(s14[:, [0, 2, 4, 6]]); Yt.append(np.asarray(lb)); S.append(s14)
P = np.concatenate(P); Yt = np.concatenate(Yt); S = np.concatenate(S)

# marginal sigma_i = || row_i(L) ||  with L lower-tri, Sigma = L L^T, diagonal = softplus(raw)
Mdia = 1e-9 + softplus(S[:, 1:8:2])
Mcov = S[:, 8:]                        # M21, M31, M32, M41, M42, M43
sig = np.stack([
    np.abs(Mdia[:, 0]),
    np.sqrt(Mcov[:, 0]**2 + Mdia[:, 1]**2),
    np.sqrt(Mcov[:, 1]**2 + Mcov[:, 2]**2 + Mdia[:, 2]**2),
    np.sqrt(Mcov[:, 3]**2 + Mcov[:, 4]**2 + Mcov[:, 5]**2 + Mdia[:, 3]**2),
], axis=1)

df = pd.DataFrame(P, columns=["x", "y", "cotA", "cotB"])
for i, c in enumerate(["xtrue", "ytrue", "cotAtrue", "cotBtrue"]): df[c] = Yt[:, i]
for i, c in enumerate(["sigmax", "sigmay", "sigmacotA", "sigmacotB"]): df[c] = sig[:, i]

pq = os.path.join(OUT_DIR, f"symbolic_n{N_EXPERTS}_vars.parquet")
df.to_parquet(pq)
json.dump({"labels_scale": ls.tolist()}, open(os.path.join(OUT_DIR, "labels_scale.json"), "w"))

print("pull check (want ~1.0 on all four):")
for i, v in enumerate(["x", "y", "cotA", "cotB"]):
    r = df[v + "true"].values - df[v].values
    print(f"  {v:<5} pull_std={(r/sig[:, i]).std():.3f}  sig_med={np.median(sig[:, i]):.4f}")
print("wrote", pq, "|", len(df), "rows")

pull check (want ~1.0 on all four):
  x     pull_std=1.008  sig_med=0.0582
  y     pull_std=1.005  sig_med=0.1136
  cotA  pull_std=1.004  sig_med=0.0307
  cotB  pull_std=1.018  sig_med=0.0550
wrote /depot/cms/private/users/kuang14/Smart_Pixel/symbolic_ablation_nexp4/symbolic_n4_vars.parquet | 108222 rows


In [14]:
# ---- 11. expert scalars + router usage  (robust to the AxisFix wrapper) ----
def ansatz_of(e):
    # e is an AxisFixExpert; its inner model holds the physics_ansatz layer
    inner = getattr(e, "inner", e)
    return inner.get_layer("physics_ansatz")

def scalar(ans, key):
    for w in ans.trainable_weights:
        if key in w.name:
            return np.asarray(w.numpy()).ravel()
    return None

expert_scalars = {}
for k, e in enumerate(core.experts):
    ans = ansatz_of(e)
    sc = {w.name: np.asarray(w.numpy()).tolist() for w in ans.trainable_weights}
    expert_scalars[f"expert_{k}"] = sc
    def g(key):
        v = scalar(ans, key)
        return None if v is None else np.round(v, 4)
    print(f"expert_{k}: theta_L={g('theta_L')} lorentz_scale={g('lorentz_scale')} "
          f"kA={g('sign_k_alpha')} a0A={g('sign_a0_alpha')} kB={g('sign_k_beta')} a0B={g('sign_a0_beta')}")
json.dump(expert_scalars, open(os.path.join(OUT_DIR, "expert_scalars.json"), "w"), indent=1, default=float)

# physics cross-check: does a0_beta track T*tan(theta_L)?  (skip cleanly if theta_L absent)
for k in range(N_EXPERTS):
    th = scalar(ansatz_of(core.experts[k]), "theta_L")
    if th is None:
        print(f"expert_{k}: theta_L not found -- skipping Lorentz cross-check"); continue
    print(f"expert_{k}: T*tan(theta_L) = {100.0*np.tan(th).item():+.2f} um  (compare a0_beta)")

usage, ent = None, None
if N_EXPERTS > 1:
    Wv = np.concatenate([core.route(vg[i][0]).numpy() for i in range(len(vg))])
    usage = Wv.mean(0)
    ent   = -(Wv * np.log(Wv + 1e-12)).sum(1).mean()
    print("\nrouter usage:", usage.round(3), "| mean entropy: %.3f (max %.3f)" % (ent, np.log(N_EXPERTS)))

expert_0: theta_L=None lorentz_scale=None kA=None a0A=None kB=None a0B=None
expert_1: theta_L=None lorentz_scale=None kA=None a0A=None kB=None a0B=None
expert_2: theta_L=None lorentz_scale=None kA=None a0A=None kB=None a0B=None
expert_3: theta_L=None lorentz_scale=None kA=None a0A=None kB=None a0B=None
expert_0: theta_L not found -- skipping Lorentz cross-check
expert_1: theta_L not found -- skipping Lorentz cross-check
expert_2: theta_L not found -- skipping Lorentz cross-check
expert_3: theta_L not found -- skipping Lorentz cross-check

router usage: [0.239 0.285 0.103 0.373] | mean entropy: 1.248 (max 1.386)


In [15]:
# ---- 12. summary.json ----
summary = {
    "training": "pure regression (NO teacher/KD) | phase1 MSE means -> phase2 orig-NLL -> phase3 corrected-NLL error head",
    "sign_source": "in-ansatz TimeGrad: sign=tanh(k*(a0-drift)); alpha<-Ty, beta<-Tx",
    "axis_fix": "AxisFixExpert swaps position slots (ansatz x<->y); baked into make_expert",
    "sigma_decode": "softplus diagonal, sigma_i = ||row_i(L)||, Sigma = L L^T (matches custom_loss_fixed)",
    "sign_init": SIGN_INIT,
    "variant": VARIANT, "n_experts": N_EXPERTS,
    "router_hidden": list(ROUTER_HIDDEN), "router_temp": ROUTER_TEMP,
    "total_params": int(model.count_params()),
    "per_expert_params": int(core.experts[0].count_params()),
    "router_params": int(n_router),
    "epochs_p1": len(h1.history["loss"]),
    "epochs_p2": len(h2.history["loss"]),
    "epochs_p3": len(h3.history["loss"]),
    "best_val_loss_p3": float(min(h3.history.get("val_loss", [float("inf")]))),
    "router_usage_val": (usage.tolist() if usage is not None else None),
    "router_entropy_val": (float(ent) if ent is not None else None),
}
json.dump(summary, open(os.path.join(OUT_DIR, "summary.json"), "w"), indent=1, default=float)
print(json.dumps(summary, indent=1))

{
 "training": "pure regression (NO teacher/KD) | phase1 MSE means -> phase2 orig-NLL -> phase3 corrected-NLL error head",
 "sign_source": "in-ansatz TimeGrad: sign=tanh(k*(a0-drift)); alpha<-Ty, beta<-Tx",
 "axis_fix": "AxisFixExpert swaps position slots (ansatz x<->y); baked into make_expert",
 "sigma_decode": "softplus diagonal, sigma_i = ||row_i(L)||, Sigma = L L^T (matches custom_loss_fixed)",
 "sign_init": {
  "initial_sign_k_alpha": 0.6288598782498782,
  "initial_sign_a0_alpha": 0.037571641135939395,
  "initial_sign_k_beta": 0.08362608149975878,
  "initial_sign_a0_beta": -43.42859471632266
 },
 "variant": "barycenter",
 "n_experts": 4,
 "router_hidden": [
  32,
  16
 ],
 "router_temp": 1.0,
 "total_params": 3268,
 "per_expert_params": 372,
 "router_params": 1780,
 "epochs_p1": 150,
 "epochs_p2": 200,
 "epochs_p3": 120,
 "best_val_loss_p3": -5.567418098449707,
 "router_usage_val": [
  0.23881801962852478,
  0.28509649634361267,
  0.10298728942871094,
  0.37309765815734863
 ],
 "r